In [1]:
from torch.utils.data import Dataset
from torchvision.io import decode_image
import os
import pandas as pd

class ImageDataset(Dataset):
    def __init__(self, annotation_file, image_dir, transform=None):
        self.img_labels = pd.read_csv(annotation_file)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, "{0:03}".format(self.img_labels.iloc[idx, 0]))
        img_path = img_path + '.png'
        image = decode_image(img_path)
        label = self.img_labels.iloc[idx, 1]
        if self.transform:
            image = self.transform(image)
        return image, label

In [2]:
import torchvision
import torch

transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize((224, 224)),
    torchvision.transforms.ConvertImageDtype(torch.float),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
    

dataset =ImageDataset(annotation_file='labels.csv', image_dir='head_ct/head_ct', transform=transform)

img, label = dataset[0]
print(f"Image shape: {img.shape}, Label: {label}")

Image shape: torch.Size([3, 224, 224]), Label: 1


In [3]:
from torch.utils.data import random_split

trainsize = int(0.8 * len(dataset))
testsize = len(dataset) - trainsize
train_dataset, test_dataset = random_split(dataset, [trainsize, testsize])

In [4]:
import torch
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)

In [5]:
from torchvision.models import resnet18
import torch.nn as nn

model = resnet18(weights="IMAGENET1K_V1")

print(model)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [13]:
feature_extractor = nn.Sequential(
    model.conv1,
    model.bn1,
    model.relu,
    model.maxpool,
    model.layer1,
    model.layer2,
    nn.AdaptiveAvgPool2d((1, 1))
)

input_tensor = train_loader.dataset[0][0].unsqueeze(0)  
print(input_tensor.shape)
features = feature_extractor(input_tensor)
print(f"Extracted features shape: {features.shape}")
features = features.view(features.size(0), -1)
print(f"Flattened features shape: {features.shape}")
features = features.squeeze(0)
print(f"Final feature vector shape: {features.shape}")
features = features.detach().numpy()
print(f"Feature vector: {features}")

torch.Size([1, 3, 224, 224])
Extracted features shape: torch.Size([1, 128, 1, 1])
Flattened features shape: torch.Size([1, 128])
Final feature vector shape: torch.Size([128])
Feature vector: [0.13678493 0.17732951 0.132548   0.09282114 0.18850556 0.13292643
 0.11518496 0.23953171 0.16887645 0.2930337  0.09879029 0.12248652
 0.10417633 0.18032415 0.17529412 0.1641147  0.17080785 0.15163699
 0.22209895 0.21249759 0.1243064  0.15863314 0.11414192 0.16774698
 0.15021424 0.23596923 0.15172127 0.18918222 0.19839947 0.2195902
 0.1687947  0.23815611 0.21485324 0.13206364 0.11233398 0.11521804
 0.35719022 0.1700852  0.20869868 0.32084024 0.24402334 0.32660013
 0.17297606 0.29680684 0.17177224 0.21395552 0.18855512 0.17978401
 0.26962724 0.12244288 0.11923132 0.12544005 0.17762655 0.11934588
 0.25150752 0.20351885 0.14800704 0.14938025 0.3948877  0.1791382
 0.21314073 0.10675504 0.13239922 0.20669077 0.14368775 0.12763995
 0.36279708 0.29979905 0.14199176 0.12663591 0.12599167 0.09493358
 0.1611